In [9]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bmadushanirodrigo/fracture-multi-region-x-ray-data",output_dir="/kaggle",
                                  force_download=True)

print("Path to dataset files:", path)
DATASET_PATH="/kaggle/input/fracture-multi-region-x-ray-data/Bone_Fracture_Binary_Classification/Bone_Fracture_Binary_Classification"


Using Colab cache for faster access to the 'fracture-multi-region-x-ray-data' dataset.
Path to dataset files: /kaggle/input/fracture-multi-region-x-ray-data


In [11]:
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader
from PIL import ImageFile
transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.transforms.ToTensor()      ]
)
ImageFile.LOAD_TRUNCATED_IMAGES = True
train_data=ImageFolder(
    f"{DATASET_PATH}/train",
    transform=transform
)
train_loader=DataLoader(
    train_data,
    batch_size=32,
    shuffle=True
)

validation_data=ImageFolder(
    f"{DATASET_PATH}/val",
    transform=transform
)
validation_loader=DataLoader(
    validation_data,
    batch_size=32,
    shuffle=True
)

test_data=ImageFolder(
    f"{DATASET_PATH}/test",
    transform=transform
)
test_loader=DataLoader(
    validation_data,
    batch_size=32,
    shuffle=True
)

In [12]:
import torch
import torch.nn as nn

class BasicBlock(nn.Module):
  def __init__(self,in_channels,out_channels,stride=1):
    super().__init__()
    if in_channels!=out_channels or stride!=1:
      self.downsample=nn.Sequential(
          nn.Conv2d(
              in_channels=in_channels,
              out_channels=out_channels,
              kernel_size=1,
              bias=False,
              stride=stride
          ),
          nn.BatchNorm2d(out_channels)
      )
    else:
      self.downsample=None
    self.conv1=nn.Conv2d(
        in_channels=in_channels,
        kernel_size=3,
        padding=1,
        stride=stride,
        bias=False,
        out_channels=out_channels
        )
    self.relu=nn.ReLU()
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.conv2=nn.Conv2d(
         in_channels=out_channels,
        kernel_size=3,
        padding=1,
        stride=1,
        bias=False,
        out_channels=out_channels
    )
    self.bn2 = nn.BatchNorm2d(out_channels)
  def forward(self,x): #conv-bn-relu-conv-bn-res-relu
      identity=x
      if self.downsample:
        identity=self.downsample(x)
      out=self.conv1(x)
      out=self.bn1(out)
      out=self.relu(out)
      out=self.conv2(out)
      out=self.bn2(out)
      out+=identity
      out=self.relu(out)
      return out

class ResNet18(nn.Module):
  def __init__(self,num_classes):
    super().__init__()
    self.conv1=nn.Conv2d(3,64,stride=2,padding=3,kernel_size=7,bias=False)
    self.bn1=nn.BatchNorm2d(64)
    self.relu=nn.ReLU()
    self.maxpool=nn.MaxPool2d(3,stride=2,padding=1)

    self.layer1=nn.Sequential(
        BasicBlock(64,64,stride=1),
        BasicBlock(64,64,stride=1),
    )
    self.layer2=nn.Sequential(
        BasicBlock(64,64,stride=2),
        BasicBlock(64,128,stride=1),
    )
    self.layer3=nn.Sequential(
        BasicBlock(128,128,stride=2),
        BasicBlock(128,256,stride=1),
    )
    self.layer4=nn.Sequential(
        BasicBlock(256,256,stride=2),
        BasicBlock(256,512,stride=1),
    )


    self.avgpool=nn.AdaptiveAvgPool2d((1,1))

    self.fc=nn.Linear(512,num_classes)
  def forward(self,x):
    out=self.conv1(x)
    out=self.bn1(out)
    out=self.relu(out)
    out=self.maxpool(out)

    out=self.layer1(out)
    out=self.layer2(out)
    out=self.layer3(out)
    out=self.layer4(out)
    out=self.avgpool(out)
    out=torch.flatten(out,1)
    out=self.fc(out)
    return out


In [ ]:
import os


epoches=30
lr=1e-3
loss_fn=torch.nn.CrossEntropyLoss()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ResNet18(2).to(device)
optim = torch.optim.Adam(model.parameters(), lr=lr)
start_epoch=0
if os.path.exists("checkpoint.pth"):
  checkpoint = torch.load("checkpoint.pth")
  model.load_state_dict(checkpoint["model_state_dict"])
  optim.load_state_dict(checkpoint["optimizer_state_dict"])
  start_epoch = checkpoint["epoch"] + 1

for epoch in range(start_epoch, start_epoch + 30):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for image, label in train_loader:

        image = image.to(device)
        label = label.to(device)

        optim.zero_grad()

        output = model(image)

        loss = loss_fn(output, label)

        loss.backward()
        optim.step()

        total_loss += loss.item()

        pred = output.argmax(dim=1)

        correct += (pred == label).sum().item()

        total += label.size(0)

    avg_loss = total_loss / len(train_loader)
    accuracy = correct / total
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optim.state_dict(),
        "loss": loss.item(),
    }, "checkpoint.pth")
    print(
        f"Epoch {epoch+1}/{epoches} | "
        f"Loss: {avg_loss:.4f} | "
        f"Acc: {accuracy:.4f}"
    )

Epoch 1/30 | Loss: 0.4296 | Acc: 0.8083
Epoch 2/30 | Loss: 0.1803 | Acc: 0.9305
Epoch 3/30 | Loss: 0.1049 | Acc: 0.9634
Epoch 4/30 | Loss: 0.0814 | Acc: 0.9709
Epoch 5/30 | Loss: 0.0767 | Acc: 0.9732
Epoch 6/30 | Loss: 0.0324 | Acc: 0.9886
Epoch 7/30 | Loss: 0.0463 | Acc: 0.9854
Epoch 8/30 | Loss: 0.0336 | Acc: 0.9890
Epoch 9/30 | Loss: 0.0332 | Acc: 0.9889
Epoch 10/30 | Loss: 0.0309 | Acc: 0.9903
Epoch 11/30 | Loss: 0.0446 | Acc: 0.9846
Epoch 12/30 | Loss: 0.0242 | Acc: 0.9911
Epoch 13/30 | Loss: 0.0188 | Acc: 0.9932
Epoch 14/30 | Loss: 0.0145 | Acc: 0.9942
Epoch 15/30 | Loss: 0.0209 | Acc: 0.9920
Epoch 16/30 | Loss: 0.0303 | Acc: 0.9897
Epoch 17/30 | Loss: 0.0152 | Acc: 0.9945
Epoch 18/30 | Loss: 0.0093 | Acc: 0.9971
Epoch 19/30 | Loss: 0.0049 | Acc: 0.9984
Epoch 20/30 | Loss: 0.0084 | Acc: 0.9970
Epoch 21/30 | Loss: 0.0267 | Acc: 0.9911
Epoch 22/30 | Loss: 0.0184 | Acc: 0.9946
Epoch 23/30 | Loss: 0.0256 | Acc: 0.9917
Epoch 24/30 | Loss: 0.0055 | Acc: 0.9983
Epoch 25/30 | Loss: 0.009

In [ ]:
torch.save(model.state_dict(),"resnet18_fractured.pth")


In [ ]:
model_loaded=ResNet18(2).to(device)
model_loaded.load_state_dict(torch.load("resnet18_fractured.pth"))
model_loaded.eval().to(device)


In [ ]:

y_true=[]
y_pred=[]
with torch.no_grad():
    for images, labels in validation_loader:
        images = images.to(device)
        labels = labels.to(device)
        pred=model_loaded(images).argmax(dim=1)
        y_true.append(labels.cpu())
        y_pred.append(pred.cpu())
y_true=torch.cat(y_true)
y_pred=torch.cat(y_pred)


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred)
rec = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
print(f"Accuracy: {acc}")
print(f"Precision: {prec}")
print(f"Recall: {rec}")
print(f"F1-score: {f1}")
print(cm)

In [ ]:
import os

print(os.getcwd())